<h1>Data Project Part A</h1>
<h3> Name: Daniel Tugendheft </h3>
<h3> ID: 318465291</h3>

<h3> #הבאת הספריות הרלוונטיות </h3>

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json

<h3> #הבאת הנתונים מהאתר והגדרות </h3>

In [5]:
url ='https://www.ad.co.il/nadlanrent?sp275=17448'
html = requests.get(url)
soup = BeautifulSoup(html.content,'html.parser')
BASE_URL = "https://www.ad.co.il"

<h3> #הגדרת פונקציות </h3>

In [7]:
# איסוף אתרים של הדירות מהדף הראשי
def get_urls_lst(listings_page_url,BASE_URL):
    
    from urllib.parse import urljoin
    
    resp = requests.get(listings_page_url)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    # חיפוש הקישורים בדף
    Nsoup = soup.find('main')
    links = Nsoup.select('a[href^="/ad/"]')

    urls = [urljoin("https://www.ad.co.il", a["href"]) for a in links]
    return list(urls)  

In [8]:
# איסוף נתונים מכל אתר
def extract_df_from_urls(lst_urls):
    results = []
    for url in lst_urls:
        resp = requests.get(url)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        # מחיר מהכותרת
        header = soup.select("div.d-flex.justify-content-between h2.card-title")
        price = header[1].get_text(strip=True) if len(header) > 1 else None
        record_data = {
        "מחיר": price
        }

        # פרטים מהטבלה
        details = {}
        table = soup.select_one("table.table.table-sm.mb-4")
        if table:
            for tr in table.select("tr"):    
                tds = tr.find_all("td")
                if len(tds) == 2:
                    key = tds[0].get_text(strip=True)
                    value = tds[1].get_text(strip=True)
                    details[key] = value

        # פרטים מהתכונות
        features = {}
        for card_icon in soup.select("div.card-icons div.card-icon"):
            feature_name = card_icon.find("span").get_text(strip=True)
            has_feature = 0 if 'disabled' in card_icon.get("class", []) else 1
            features[feature_name] = has_feature

        # פרטי תיאור ומספר תמונות שיש להוסיף
        description_tag = soup.select_one("div.p-3 p.text-word-break")
        description = description_tag.get_text(strip=True) if description_tag else None
        num_of_pic = len(soup.select('div[itemtype="http://schema.org/ImageGallery"] a[itemprop="contentUrl"]'))
        more_data = {
        "תיאור": description,
        "מספר תמונות": num_of_pic
        }
        
        # איחוד לשורה והכנסה לתוצאות
        record_data.update(details)
        record_data.update(features)
        record_data.update(more_data)
        results.append(record_data)

    df = pd.DataFrame(results)
    
    # הסרת עמודות מיותרות לפרויקט 
    df = df.drop(columns=["אזור", "עיר", "על עמודים", "מרפסת שמש", "מרפסות"])
    
    # הסרת רווחים והחלפת קרקע ב-0
    df["קומה"] = df["קומה"].str.strip().str.replace("קרקע", "0", regex=False)

    # פיצול לפי קומה וקומות בבניין
    df[["קומה", "קומות בבניין"]] = df["קומה"].str.extract(r"(\d+)\s*מתוך\s*(\d+)").astype("Int64")

    # טיפול בתאריך כניסה
    df['תאריך כניסה'] = df['תאריך כניסה'].fillna('מיידית').replace('מיידית', 0).astype(int)

    return df


In [9]:
# בדיקת מרחק ממרכז העיר (api)
def get_distance_from_center(origin):
    url = "https://maps.googleapis.com/maps/api/distancematrix/json"
    api_key = 'AIzaSy...' #your_api_key
    params = {
        "origins": origin,
        "destinations": 'קניון רחובות, רחובות, ישראל',
        "key": api_key,
        "language": "he",  # עברית
        "units": "metric"  # מטרים/ק"מ
    }

    try:
        response = requests.get(url, params=params)
        if not response.status_code == 200:
            print("HTTP error", response. status_code)
            return None
        else:
            try:
                data = response.json()
                distance_value = data["rows"][0]["elements"][0]["distance"]["value"]  #במטרים
                return distance_value
            except:
                print ("Response not in valid JSON format")
    except:
        print ("Something went wrong with requests.get")

In [10]:
# הרצת כתובות לעמודת מרחק מהמרכז
def add_distances_column(df):
    #בדיקה קלה שהכתובת רשומה והוצאת מרחק
    def safe_get_distance(address):
        if pd.isna(address) or address.strip() == "":
            return None
        return get_distance_from_center(f"{address}, רחובות")

    df["distance_from_center"] = df['כתובת'].apply(safe_get_distance)
    return df

<h3> #יישום הפונקציות לבניית טבלה </h3>

In [12]:
lst_urls = get_urls_lst(url,BASE_URL)
df = extract_df_from_urls(lst_urls)
add_distances_column(df)
df.head(3)

/var/folders/t9/ydjt2dtn5t704kwr5dxgkprm0000gn/T/ipykernel_18734/1134338154.py:61: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['תאריך כניסה'] = df['תאריך כניסה'].fillna('מיידית').replace('מיידית', 0).astype(int)


,מחיר,פרטי הנכס,שכונה,כתובת,חדרים,קומה,שטח בנוי,תאריך כניסה,תשלומים בשנה,ארנונה בחודש,...,סורגים,מעלית,מחסן,משופצת,תיאור,מספר תמונות,ועד בית בחודש,שטח גינה,קומות בבניין,distance_from_center
0,"3,200 ₪",דירה,קרית משה,הרב נדב דוד 11,3,4,70,0,12,250,...,0,0,0,0,"שכונה קרית משה\n\nטלוויזיה, מקרר, מזגן, תנור ו...",9,NaN,NaN,4,3697
1,"4,200 ₪",דירה,"ביל""ו",מרשוב 4,3.5,4,90,0,12,1300,...,0,0,0,0,"ללא תיווך בהזדמנות, ברחובות סמוך לקניון דירת 3...",9,150,NaN,4,1691
2,"5,000 ₪",דירה,NaN,הרצל 29,4,4,86,0,12,380,...,0,0,0,0,"דירת 4 חדרים עם 2 חדרי שירותים משופצים, מטבח מ...",4,180,NaN,8,2098


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 26 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   מחיר                  42 non-null     object
 1   פרטי הנכס             42 non-null     object
 2   שכונה                 29 non-null     object
 3   כתובת                 42 non-null     object
 4   חדרים                 42 non-null     object
 5   קומה                  42 non-null     Int64 
 6   שטח בנוי              42 non-null     object
 7   תאריך כניסה           42 non-null     int64 
 8   תשלומים בשנה          42 non-null     object
 9   ארנונה בחודש          25 non-null     object
 10  מרוהטת                42 non-null     int64 
 11  מזגן                  42 non-null     int64 
 12  חניה                  42 non-null     int64 
 13  ממ"ד                  42 non-null     int64 
 14  מרפסת                 42 non-null     int64 
 15  נגישות                42 non-null     int6

<h3> # תקנון הטבלה לפורמט הרצוי להמשך עבודה  </h3>

In [15]:
#התאמת הדאטה פריים לסטנדרט המתבקש
Ndf = df
column_mapping = {
    "פרטי הנכס": "property_type",
    "שכונה": "neighborhood",
    "כתובת": "address",
    "חדרים": "room_num",
    "קומה": "floor",
    "שטח בנוי": "area",
    "שטח גינה": "garden_area",
    "תאריך כניסה": "days_to_enter",
    "תשלומים בשנה": "num_of_payments",
    "ארנונה בחודש": "monthly_arnona",
    "ועד בית בחודש": "building_tax",
    "קומות בבניין": "total_floors",
    "תיאור": "description",
    "חניה": "has_parking",
    "מחסן": "has_stotsge",
    "מעלית": "elevator",
    "מזגן": "ac",
    "נגישות": "handicap",
    "סורגים": "has_bars",
    "ממ\"ד": "has_safe_room",
    "מרפסת": "has_balcon",
    "מרוהטת": "is_furnished",
    "משופצת": "is_renovated",
    "מחיר": "price",
    "מספר תמונות": "num_of_images",
    "distance_from_center": "distance_from_center"
}

#טיפול במחיר
Ndf['מחיר'] = Ndf['מחיר'].str.replace('[₪,]', '', regex=True).astype(float)

#טיפול בערכים חסרים בועד בית לבתי קרקע(אין ועד)
mask = ~Ndf['פרטי הנכס'].isin(['דירה', 'דירה להשכרה']) & Ndf['ועד בית בחודש'].isna()
Ndf.loc[mask, 'ועד בית בחודש'] = 0
Ndf['ועד בית בחודש'] = pd.to_numeric(Ndf['ועד בית בחודש'], errors='coerce')

#טיפול בערכים חסרים בשטח גינה(אין גינה)
Ndf['שטח גינה'] = pd.to_numeric(Ndf['שטח גינה'], errors='coerce').fillna(0)

# מיפוי שמות עמודות לאנגלית וסידור לפי סדר
Ndf.rename(columns=column_mapping, inplace=True)
Ndf = Ndf[list(column_mapping.values())]

# טיפוסים
Ndf['property_type'] = Ndf['property_type'].astype('string')
Ndf['neighborhood'] = Ndf['neighborhood'].astype('string')
Ndf['address'] = Ndf['address'].astype('string')
Ndf['room_num'] = Ndf['room_num'].astype('float')
Ndf['floor'] = Ndf['floor'].astype('Int64')
Ndf['area'] = Ndf['area'].astype('Int64')
Ndf['garden_area'] = Ndf['garden_area'].astype('Int64')
Ndf['days_to_enter'] = Ndf['days_to_enter'].astype('Int64')
Ndf['num_of_payments'] = Ndf['num_of_payments'].astype('Int64')
Ndf['monthly_arnona'] = Ndf['monthly_arnona'].astype('Int64')
Ndf['building_tax'] = Ndf['building_tax'].astype('Int64')
Ndf['total_floors'] = Ndf['total_floors'].astype('Int64')
Ndf['description'] = Ndf['description'].astype('string')
Ndf['has_parking'] = Ndf['has_parking'].astype('Int64')
Ndf['has_stotsge'] = Ndf['has_stotsge'].astype('Int64')
Ndf['elevator'] = Ndf['elevator'].astype('Int64')
Ndf['ac'] = Ndf['ac'].astype('Int64')
Ndf['handicap'] = Ndf['handicap'].astype('Int64')
Ndf['has_bars'] = Ndf['has_bars'].astype('Int64')
Ndf['has_safe_room'] = Ndf['has_safe_room'].astype('Int64')
Ndf['has_balcon'] = Ndf['has_balcon'].astype('Int64')
Ndf['is_furnished'] = Ndf['is_furnished'].astype('Int64')
Ndf['is_renovated'] = Ndf['is_renovated'].astype('Int64')
Ndf['price'] = Ndf['price'].astype('float')
Ndf['num_of_images'] = Ndf['num_of_images'].astype('Int64')
Ndf['distance_from_center'] = Ndf['distance_from_center'].astype('float')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 26 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   price                 42 non-null     float64
 1   property_type         42 non-null     object 
 2   neighborhood          29 non-null     object 
 3   address               42 non-null     object 
 4   room_num              42 non-null     object 
 5   floor                 42 non-null     Int64  
 6   area                  42 non-null     object 
 7   days_to_enter         42 non-null     int64  
 8   num_of_payments       42 non-null     object 
 9   monthly_arnona        25 non-null     object 
 10  is_furnished          42 non-null     int64  
 11  ac                    42 non-null     int64  
 12  has_parking           42 non-null     int64  
 13  has_safe_room         42 non-null     int64  
 14  has_balcon            42 non-null     int64  
 15  handicap              42 

<h3> #טבלת הדאטה הסופית </h3>

In [17]:
Ndf

,property_type,neighborhood,address,room_num,floor,area,garden_area,days_to_enter,num_of_payments,monthly_arnona,...,ac,handicap,has_bars,has_safe_room,has_balcon,is_furnished,is_renovated,price,num_of_images,distance_from_center
0,דירה,קרית משה,הרב נדב דוד 11,3.0,4,70,0,0,12,250,...,0,0,0,0,0,0,0,3200.0,9,3697.0
1,דירה,"ביל""ו",מרשוב 4,3.5,4,90,0,0,12,1300,...,1,0,0,0,1,0,0,4200.0,9,1691.0
2,דירה,<NA>,הרצל 29,4.0,4,86,0,0,12,380,...,0,0,0,0,0,0,0,5000.0,4,2098.0
3,דו משפחתי,שמורת מליבו,זמל זוסיא 6,6.0,0,160,250,0,12,750,...,0,0,0,0,1,0,0,15000.0,9,6046.0
4,דירה,"ביל""ו",פינס 5,4.0,4,110,0,0,12,600,...,1,1,0,0,0,0,0,5000.0,9,1691.0
5,דירה,בנימין;נורדאו,מנדלי מוכר ספרים 15,2.5,0,45,0,0,12,569,...,0,0,0,0,1,0,0,4500.0,7,1449.0
6,מרתף/פרטר,גני איריס ; גבעת האירוסים,פעמונית 1,2.0,0,65,0,0,12,<NA>,...,1,0,0,1,0,1,0,4100.0,5,1691.0
7,סטודיו/לופט,<NA>,אשכול 2,1.0,1,40,0,0,12,<NA>,...,1,0,0,0,0,1,0,2900.0,7,9860.0
8,יחידת דיור,<NA>,אשכול 2,2.0,1,50,0,0,12,<NA>,...,0,0,0,0,0,0,0,3600.0,8,9860.0
9,דירה,שכונת יחד,גורדון 7,4.5,1,110,0,0,12,1200,...,0,0,0,1,0,1,0,6000.0,12,4725.0


In [18]:
Ndf.to_csv('AprtRehovot.csv', index=False, encoding='utf-8-sig')